<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(
    title="Production-Ready AI API",
    description="Day 21 AI API",
    version="1.0.0"
)


# Request Model
class AIRequest(BaseModel):
    query: str = Field(
        ...,
        description="User query for the AI system",
        examples=["Explain machine learning in simple terms"]
    )


# Response Model
class AIResponse(BaseModel):
    answer: str = Field(
        ...,
        description="Generated answer from the AI system",
        examples=["Machine learning allows computers to learn from data."]
    )


@app.post("/ask", response_model=AIResponse)
async def ask_ai(request: AIRequest):
    return AIResponse(
        answer=f"You asked: {request.query}"
    )

In [2]:
from pydantic import BaseModel, Field, field_validator
import re


class AIRequest(BaseModel):
    query: str = Field(
        ...,
        description="User query for the AI system",
        examples=["Explain machine learning in simple terms"]
    )

    @field_validator("query")
    @classmethod
    def validate_query(cls, value: str) -> str:

        # Remove leading/trailing spaces
        value = value.strip()

        # Check minimum length
        if len(value) < 5:
            raise ValueError(
                "INPUT_INVALID: Query must contain at least 5 characters"
            )

        # Check maximum length
        if len(value) > 1000:
            raise ValueError(
                "INPUT_INVALID: Query must not exceed 1000 characters"
            )

        # Reject whitespace or special-character-only input
        if not re.search(r"[A-Za-z0-9]", value):
            raise ValueError(
                "INPUT_INVALID: Query must contain letters or numbers"
            )

        return value

In [3]:
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator
import re
import uuid

In [9]:
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(
    request: Request,
    exc: RequestValidationError
):
    request_id = str(uuid.uuid4())

    errors = exc.errors()

    message = "Invalid request"

    if errors:
        message = errors[0].get("msg", "Invalid request")

        if message.startswith("Value error, "):
            message = message.replace("Value error, ", "", 1)

        if message.startswith("INPUT_INVALID:"):
            message = message.replace("INPUT_INVALID: ", "", 1)

    return JSONResponse(
        status_code=422,
        content={
            "code": "INPUT_INVALID",
            "message": message,
            "request_id": request_id
        }
    )

In [5]:
class AIRequest(BaseModel):
    query: str = Field(
        ...,
        description="User query for the AI system",
        examples=["Explain machine learning in simple terms"]
    )

    @field_validator("query")
    @classmethod
    def validate_query(cls, value: str) -> str:

        value = value.strip()

        if len(value) < 5:
            raise ValueError(
                "INPUT_INVALID: Query must contain at least 5 characters"
            )

        if len(value) > 1000:
            raise ValueError(
                "INPUT_INVALID: Query must not exceed 1000 characters"
            )

        if not re.search(r"[A-Za-z0-9]", value):
            raise ValueError(
                "INPUT_INVALID: Query must contain letters or numbers"
            )

        return value

In [6]:
from fastapi import FastAPI, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse

In [7]:
from pydantic import BaseModel, Field, field_validator
import re
import uuid

In [10]:
class ErrorResponse(BaseModel):
    code: str = Field(
        ...,
        description="Structured error code",
        examples=["INPUT_INVALID"]
    )

    message: str = Field(
        ...,
        description="Human-readable error message",
        examples=["Query must contain at least 5 characters"]
    )

    request_id: str = Field(
        ...,
        description="Unique ID for tracking the request",
        examples=["550e8400-e29b-41d4-a716-446655440000"]
    )

In [11]:
class RetrievalError(Exception):
    pass


async def retrieve_documents(query: str):
    # Temporary simulation of a retrieval system
    # We will replace this with the real retrieval pipeline later.

    if query == "trigger retrieval failure":
        raise RetrievalError("Document retrieval failed")

    return ["Relevant document 1", "Relevant document 2"]

In [12]:
@app.exception_handler(RetrievalError)
async def retrieval_error_handler(
    request: Request,
    exc: RetrievalError
):
    request_id = str(uuid.uuid4())

    return JSONResponse(
        status_code=500,
        content={
            "code": "RETRIEVAL_FAILURE",
            "message": str(exc),
            "request_id": request_id
        }
    )

In [13]:
@app.post("/ask", response_model=AIResponse)
async def ask_ai(request: AIRequest):

    documents = await retrieve_documents(request.query)

    return AIResponse(
        answer=f"Retrieved {len(documents)} documents for: {request.query}"
    )

In [14]:
import asyncio
from openai import RateLimitError

In [15]:
MAX_RETRIES = 3
INITIAL_BACKOFF = 1

In [16]:
async def generate_with_retry(query: str):

    for attempt in range(MAX_RETRIES + 1):

        try:
            # Temporary simulation
            # We will connect the real OpenAI API later.
            if query == "trigger rate limit":
                raise RateLimitError(
                    message="Rate limit exceeded",
                    response=None,
                    body=None
                )

            return f"AI response for: {query}"

        except RateLimitError:

            if attempt == MAX_RETRIES:
                raise

            wait_time = INITIAL_BACKOFF * (2 ** attempt)

            print(
                f"Rate limit detected. "
                f"Retry {attempt + 1}/{MAX_RETRIES} "
                f"in {wait_time} seconds..."
            )

            await asyncio.sleep(wait_time)

In [17]:
@app.get("/test-retry")
async def test_retry():

    try:
        result = await generate_with_retry("trigger rate limit")

        return {
            "status": "success",
            "result": result
        }

    except RateLimitError:
        return {
            "status": "failed",
            "message": "Maximum retry attempts reached"
        }

In [20]:
import os
print("Current directory:", os.getcwd())
print("Files:")
print(os.listdir())

Current directory: /content
Files:
['.config', 'sample_data']


In [22]:
%%writefile app.py

from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "FastAPI is working"}

Writing app.py


In [23]:
import os
print(os.path.exists("app.py"))

True


In [24]:
!uvicorn app:app --host 127.0.0.1 --port 8000

INFO:     Started server process [5815]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [5815]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "uvloop/loop.pyx", line 1512, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1505, in uvloop.loop.Loop.run_until_complete
    self.run_forever()
  File "uvloop/loop.pyx", line 1379, in uvloop.loop.Loop.run_forever
    self._run(mode)
  File "uvloop/loop.pyx", line 557, in uvloop.loop.Loop._run
    raise self._last_error
  File "uvloop/loop.pyx", line 476, in uvloop.loop.Loop._on

In [25]:
from fastapi.responses import StreamingResponse
import asyncio

In [26]:
async def generate_stream(query: str):

    response = await generate_with_retry(query)

    words = response.split()

    for word in words:
        yield word + " "
        await asyncio.sleep(0.2)

In [27]:
@app.get("/stream")
async def stream_response(query: str):

    return StreamingResponse(
        generate_stream(query),
        media_type="text/plain"
    )

In [31]:
!uvicorn app:app --host 127.0.0.1 --port 8000

INFO:     Started server process [9971]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Finished server process [9971]
ERROR:    Traceback (most recent call last):
  File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/asyncio/runners.py", line 119, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "uvloop/loop.pyx", line 1512, in uvloop.loop.Loop.run_until_complete
  File "uvloop/loop.pyx", line 1505, in uvloop.loop.Loop.run_until_complete
    self.run_forever()
  File "uvloop/loop.pyx", line 1379, in uvloop.loop.Loop.run_forever
    self._run(mode)
  File "uvloop/loop.pyx", line 557, in uvloop.loop.Loop._run
    raise self._last_error
  File "uvloop/loop.pyx", line 476, in uvloop.loop.Loop._on

In [35]:
import subprocess
import time

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"]
)

time.sleep(2)

print("Server started")

Server started


In [36]:
import requests

response = requests.get("http://127.0.0.1:8000/docs")

print(response.status_code)

200


In [37]:
import requests

with requests.get(
    "http://127.0.0.1:8000/stream",
    params={"query": "Explain artificial intelligence"},
    stream=True
) as response:

    print("Status:", response.status_code)

    for chunk in response.iter_content(chunk_size=None):
        if chunk:
            print(chunk.decode(), end="", flush=True)

Status: 404
{"detail":"Not Found"}

In [38]:
REQUEST_TIMEOUT = 15

In [39]:
async def generate_with_timeout(query: str):

    try:
        result = await asyncio.wait_for(
            generate_with_retry(query),
            timeout=REQUEST_TIMEOUT
        )

        return result

    except asyncio.TimeoutError:
        raise TimeoutError("LLM generation timed out")

In [40]:
@app.exception_handler(TimeoutError)
async def timeout_error_handler(
    request: Request,
    exc: TimeoutError
):
    request_id = str(uuid.uuid4())

    return JSONResponse(
        status_code=504,
        content={
            "code": "LLM_TIMEOUT",
            "message": str(exc),
            "request_id": request_id
        }
    )

In [41]:
async def generate_with_retry(query: str):

    for attempt in range(MAX_RETRIES + 1):

        try:

            if query == "trigger timeout":
                await asyncio.sleep(20)

            return f"AI response for: {query}"

        except RateLimitError:

            if attempt == MAX_RETRIES:
                raise

            wait_time = INITIAL_BACKOFF * (2 ** attempt)

            print(
                f"Rate limit detected. "
                f"Retry {attempt + 1}/{MAX_RETRIES} "
                f"in {wait_time} seconds..."
            )

            await asyncio.sleep(wait_time)

In [42]:
@app.get("/test-timeout")
async def test_timeout():

    result = await generate_with_timeout("trigger timeout")

    return {
        "status": "success",
        "result": result
    }

In [43]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/test-timeout"
)

print("Status:", response.status_code)
print(response.json())

Status: 404
{'detail': 'Not Found'}


In [44]:
{
    "code": "LLM_TIMEOUT",
    "message": "LLM generation timed out",
    "request_id": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
}

{'code': 'LLM_TIMEOUT',
 'message': 'LLM generation timed out',
 'request_id': 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'}

In [45]:
async def generate_stream(query: str):

    try:
        response = await generate_with_timeout(query)

        words = response.split()

        for word in words:
            yield word + " "
            await asyncio.sleep(0.2)

    except TimeoutError as exc:
        yield str({
            "code": "LLM_TIMEOUT",
            "message": str(exc),
            "request_id": str(uuid.uuid4())
        })

In [46]:
@app.get("/stream")
async def stream_response(query: str):

    return StreamingResponse(
        generate_stream(query),
        media_type="text/plain"
    )

In [47]:
@app.get("/stream")
async def stream_response(query: str):

    return StreamingResponse(
        generate_stream(query),
        media_type="text/plain"
    )

In [49]:
for route in app.routes:
    print(route.path, route.methods)

/openapi.json {'GET', 'HEAD'}
/docs {'GET', 'HEAD'}
/docs/oauth2-redirect {'GET', 'HEAD'}
/redoc {'GET', 'HEAD'}
/ask {'POST'}
/ask {'POST'}
/test-retry {'GET'}
/stream {'GET'}
/test-timeout {'GET'}
/stream {'GET'}
/stream {'GET'}


In [50]:
@app.get("/test-timeout")
async def test_timeout():

    result = await generate_with_timeout("trigger timeout")

    return {
        "status": "success",
        "result": result
    }

In [51]:
import subprocess
import time

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"]
)

time.sleep(2)

print("Server restarted")

Server restarted


In [53]:
import os

print("Current directory:", os.getcwd())
print("app.py exists:", os.path.exists("/content/app.py"))

if os.path.exists("/content/app.py"):
    print("\n--- app.py contents ---")
    with open("/content/app.py", "r") as f:
        print(f.read())

Current directory: /content
app.py exists: True

--- app.py contents ---

from fastapi import FastAPI

app = FastAPI()

@app.get("/")
async def root():
    return {"message": "FastAPI is working"}



In [54]:
import requests

response = requests.get("http://127.0.0.1:8000/openapi.json")

print(response.status_code)
print(response.json()["paths"])

200
{'/': {'get': {'summary': 'Root', 'operationId': 'root__get', 'responses': {'200': {'description': 'Successful Response', 'content': {'application/json': {'schema': {}}}}}}}}


In [55]:
def create_error_response(
    code: str,
    message: str,
    status_code: int
):
    request_id = str(uuid.uuid4())

    return JSONResponse(
        status_code=status_code,
        content={
            "code": code,
            "message": message,
            "request_id": request_id
        }
    )

In [56]:
@app.exception_handler(RetrievalError)
async def retrieval_error_handler(
    request: Request,
    exc: RetrievalError
):
    return create_error_response(
        code="RETRIEVAL_FAILURE",
        message=str(exc),
        status_code=500
    )

In [57]:
@app.exception_handler(TimeoutError)
async def timeout_error_handler(
    request: Request,
    exc: TimeoutError
):
    return create_error_response(
        code="LLM_TIMEOUT",
        message=str(exc),
        status_code=504
    )

In [58]:
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(
    request: Request,
    exc: RequestValidationError
):
    errors = exc.errors()
    message = "Invalid request"

    if errors:
        message = errors[0].get("msg", "Invalid request")

        if message.startswith("Value error, "):
            message = message.replace("Value error, ", "", 1)

        if message.startswith("INPUT_INVALID:"):
            message = message.replace("INPUT_INVALID: ", "", 1)

    return create_error_response(
        code="INPUT_INVALID",
        message=message,
        status_code=422
    )

In [59]:
@app.post("/ask", response_model=AIResponse)
async def ask_ai(request: AIRequest):

    # Step 1: Retrieve relevant documents
    documents = await retrieve_documents(request.query)

    # Step 2: Generate AI response with retry logic
    answer = await generate_with_retry(request.query)

    # Step 3: Return structured response
    return AIResponse(
        answer=answer
    )

In [60]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "query": "Explain artificial intelligence"
    }
)

print("Status:", response.status_code)
print(response.json())

Status: 404
{'detail': 'Not Found'}


In [61]:
REQUEST_TIMEOUT = 15

In [62]:
async def generate_stream(query: str):

    try:
        response = await generate_with_timeout(query)

        for word in response.split():
            yield word + " "
            await asyncio.sleep(0.2)

    except TimeoutError as exc:
        error = {
            "code": "LLM_TIMEOUT",
            "message": str(exc),
            "request_id": str(uuid.uuid4())
        }

        yield str(error)

In [63]:
@app.get("/stream")
async def stream_response(query: str):

    return StreamingResponse(
        generate_stream(query),
        media_type="text/plain"
    )

In [64]:
import requests

with requests.get(
    "http://127.0.0.1:8000/stream",
    params={"query": "Explain artificial intelligence"},
    stream=True
) as response:

    print("Status:", response.status_code)

    for chunk in response.iter_content(chunk_size=None):
        if chunk:
            print(chunk.decode(), end="", flush=True)

Status: 404
{"detail":"Not Found"}

In [65]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/test-timeout"
)

print("Status:", response.status_code)
print(response.json())

Status: 404
{'detail': 'Not Found'}


In [66]:
@app.get("/test-retrieval-error")
async def test_retrieval_error():
    await retrieve_documents("trigger retrieval failure")
    return {"status": "success"}


@app.get("/test-llm-timeout")
async def test_llm_timeout():
    result = await generate_with_timeout("trigger timeout")
    return {"status": "success", "result": result}

In [67]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/test-retrieval-error"
)

print("Status:", response.status_code)
print("Body:", response.json())

Status: 404
Body: {'detail': 'Not Found'}


In [68]:
response = requests.get(
    "http://127.0.0.1:8000/test-llm-timeout"
)

print("Status:", response.status_code)
print("Body:", response.json())

Status: 404
Body: {'detail': 'Not Found'}


In [69]:
response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"query": "AI"}
)

print("Status:", response.status_code)
print("Body:", response.json())

Status: 404
Body: {'detail': 'Not Found'}


In [70]:
@app.post("/ask", response_model=AIResponse)
async def ask_ai(request: AIRequest):

    # Step 1: Retrieve relevant documents
    documents = await retrieve_documents(request.query)

    # Step 2: Generate answer with retry + timeout
    answer = await generate_with_timeout(request.query)

    # Step 3: Return structured response
    return AIResponse(
        answer=answer
    )

In [71]:
response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "query": "Explain artificial intelligence"
    }
)

print("Status:", response.status_code)
print("Body:", response.json())

Status: 404
Body: {'detail': 'Not Found'}
